# Analyse des Dépendances Syntaxiques avec spaCy
Analyse des reviews par langue avec détection et étude des structures syntaxiques

## 1. Installation des dépendances

Dans cette section, les bibliothèques Python nécessaires au pipeline sont installées automatiquement (`pandas`, `spaCy`, `langdetect`, etc.).
L’objectif est d’assurer que l’environnement est prêt avant toute étape d’analyse.

In [1]:
# Installation des packages nécessaires
import subprocess
import sys

packages = [
    'pandas',
    'spacy',
    'langdetect',
    'matplotlib',
    'seaborn',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

print("Packages installés avec succès!")

Packages installés avec succès!


## 2. Téléchargement des modèles spaCy

Cette section télécharge les modèles spaCy multilingues utilisés ensuite pour l’annotation syntaxique.
Chaque modèle correspond à une langue ciblée dans le corpus.

In [2]:
import subprocess
import sys

# Télécharger les modèles spaCy pour les principales langues
models = {
    'fr': 'fr_core_news_sm',
    'en': 'en_core_web_sm',
    'es': 'es_core_news_sm',
    'de': 'de_core_news_sm',
    'pt': 'pt_core_news_sm',
    'it': 'it_core_news_sm'
}

for lang_code, model_name in models.items():
    print(f"Téléchargement du modèle {model_name} pour {lang_code}...")
    try:
        subprocess.check_call([sys.executable, '-m', 'spacy', 'download', model_name, '-q'])
        print(f"  ✓ {model_name} téléchargé")
    except:
        print(f"  ✗ Erreur lors du téléchargement de {model_name}")

print("\nTéléchargements terminés!")

Téléchargement du modèle fr_core_news_sm pour fr...
  ✓ fr_core_news_sm téléchargé
Téléchargement du modèle en_core_web_sm pour en...
  ✓ en_core_web_sm téléchargé
Téléchargement du modèle es_core_news_sm pour es...
  ✓ es_core_news_sm téléchargé
Téléchargement du modèle de_core_news_sm pour de...
  ✓ de_core_news_sm téléchargé
Téléchargement du modèle pt_core_news_sm pour pt...
  ✓ pt_core_news_sm téléchargé
Téléchargement du modèle it_core_news_sm pour it...
  ✓ it_core_news_sm téléchargé

Téléchargements terminés!


## 3. Imports et chargement des données

Ici, le notebook importe les modules principaux, initialise les paramètres globaux (temps, énergie, CO2, taille d’échantillon) et charge le jeu de données source.
Cette étape pose le cadre de calcul pour toutes les sections suivantes.

In [3]:
import pandas as pd
import numpy as np
import spacy
from langdetect import detect, detect_langs
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from spacy import displacy
import warnings
import time
warnings.filterwarnings('ignore')

# Paramètres globaux temps/carbone (un seul compteur pour tout le script)
POWER_WATTS = 150  # puissance machine
FR_GRID_KGCO2_PER_KWH = 0.056  # électricité française (kgCO2e/kWh)
SCRIPT_START = time.perf_counter()

# Taille d'échantillon globale
# - valeur entière: nombre de lignes traitées
# - "total": traiter tout le fichier filtré
Sample_Size = 10000
k = Sample_Size

# Configuration des visualisations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Charger les données
df = pd.read_csv('data/reviews_select.csv', index_col=0)
print(f"Données chargées: {len(df)} reviews")
print(f"\nColonnes: {df.columns.tolist()}")
print(f"\nAperçu:")
df.head()

Données chargées: 530818 reviews

Colonnes: ['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'langue', 'langue_short', 'nwords']

Aperçu:


,listing_id,id,date,reviewer_id,reviewer_name,comments,langue,langue_short,nwords
1,5396.0,9.164898e+17,2023-06-18,497745629.0,Grazia,"Alloggio confortevole e pratico, dotato di tut...",it,Latin,18.0
2,5396.0,9.216001e+17,2023-06-25,70206366.0,Benjamin,Très bon emplacement pour cet appartement typi...,fr,French,27.0
3,5396.0,9.266632e+17,2023-07-02,41320355.0,Julia,"What a wonderful gem. Great location, it was ...",en,English,28.0
4,5396.0,9.288200e+17,2023-07-05,259277676.0,GLori,We had a lovely 3 night stay. Everything was ...,en,English,31.0
5,5396.0,9.295177e+17,2023-07-06,93503104.0,James,Great location. Very calm and quiet. Small but...,en,English,31.0


## 4. Analyse de la distribution des langues

Cette partie fournit un diagnostic rapide de la distribution linguistique du corpus (codes courts et libellés de langue).
Elle permet de vérifier la couverture des langues avant l’analyse syntaxique.

In [4]:
# Résumé textuel (sans visualisation graphique)
print("Langue par code court:")
langue_dist = df['langue_short'].value_counts()
print(langue_dist)

print("\nLangue par nom:")
langue_dist_name = df['langue'].value_counts()
print(langue_dist_name.head(15))

print(f"\nTotal de {len(langue_dist)} langues différentes détectées")

Langue par code court:
langue_short
English      269778
French       129519
Latin         56380
Germanic      29852
East Asia     17494
others        16088
Name: count, dtype: int64

Langue par nom:
langue
en    269778
fr    129519
es     29932
de     22314
it     14528
pt     10204
ko      8554
nl      7538
zh      5339
ja      3601
ru      1929
da      1782
tr      1484
ca      1407
pl      1147
Name: count, dtype: int64

Total de 6 langues différentes détectées


## 5. Sélection des langues principales et chargement des modèles spaCy

La section définit le mapping langue → modèle spaCy, puis charge les modèles disponibles en mémoire.
Le résultat est un dictionnaire `nlp_models` réutilisé pour toutes les annotations.

In [5]:
# Définir les principales langues et leurs modèles spaCy
spacy_models = {
    'fr': 'fr_core_news_sm',
    'en': 'en_core_web_sm',
    'es': 'es_core_news_sm',
    'de': 'de_core_news_sm',
    'pt': 'pt_core_news_sm',
    'it': 'it_core_news_sm'
}

# Charger les modèles spaCy disponibles
nlp_models = {}
for lang_code, model_name in spacy_models.items():
    try:
        nlp_models[lang_code] = spacy.load(model_name)
        print(f"✓ Modèle chargé: {lang_code} ({model_name})")
    except:
        print(f"✗ Impossible de charger le modèle: {lang_code} ({model_name})")

print(f"\n{len(nlp_models)} modèles spaCy chargés avec succès")

✓ Modèle chargé: fr (fr_core_news_sm)
✓ Modèle chargé: en (en_core_web_sm)
✓ Modèle chargé: es (es_core_news_sm)
✓ Modèle chargé: de (de_core_news_sm)
✓ Modèle chargé: pt (pt_core_news_sm)
✓ Modèle chargé: it (it_core_news_sm)

6 modèles spaCy chargés avec succès


## 6. Analyse des dépendances syntaxiques par langue

Ici, les données sont filtrées (langues supportées, commentaires non vides), puis échantillonnées selon `Sample_Size`/`k`.
Cette étape construit `df_sample`, l’échantillon effectivement analysé.

In [6]:
# Filtrer les reviews pour les langues principales avec modèles spaCy
# IMPORTANT: on utilise la colonne `langue` (codes ISO: en, fr, es, de, pt, it)
# 1) Taille pilotée par Sample_Size (ou alias k)
# 2) Exclure les textes linguistiquement non identifiés
df_work = df.copy()
df_work['lang_code'] = df_work['langue'].astype(str).str.lower().str.strip()

# Conserver uniquement les langues prises en charge (textes identifiés)
df_filtered = df_work[df_work['lang_code'].isin(nlp_models.keys())].copy()

# Exclure textes vides / NA
mask_text_valid = df_filtered['comments'].notna() & (df_filtered['comments'].astype(str).str.strip() != '')
df_filtered = df_filtered[mask_text_valid].copy()

# Taille d'échantillon paramétrable
sample_param = k if 'k' in globals() else Sample_Size

if isinstance(sample_param, str) and sample_param.lower() == 'total':
    df_sample = df_filtered.copy().reset_index(drop=True)
    sample_label = 'total'
else:
    sample_n = int(sample_param)
    if sample_n <= 0:
        raise ValueError('Sample_Size doit être un entier > 0 ou "total"')
    df_sample = df_filtered.head(min(sample_n, len(df_filtered))).copy().reset_index(drop=True)
    sample_label = str(sample_n)

print(f"Reviews éligibles (langue identifiée + texte valide): {len(df_filtered)}")
print(f"\nSample_Size={sample_label} -> {len(df_sample)} textes traités")
print("\nDistribution des langues dans l'échantillon:")
print(df_sample['lang_code'].value_counts())

Reviews éligibles (langue identifiée + texte valide): 476269

Sample_Size=10000 -> 10000 textes traités

Distribution des langues dans l'échantillon:
lang_code
en    6680
fr    1673
es     608
de     494
it     275
pt     270
Name: count, dtype: int64


## 7. Extraction des relations de dépendance syntaxique

Cette section exécute l’annotation syntaxique par langue, extrait les relations de dépendance et agrège des statistiques POS/DEP.
Une barre de progression `tqdm` facilite le suivi du traitement.

In [7]:
# Fonction pour extraire les dépendances syntaxiques
def extract_dependencies(text, nlp):
    """Extrait les relations de dépendance d'un texte"""
    try:
        doc = nlp(text[:1000])  # Limiter à 1000 caractères
        dependencies = []

        for token in doc:
            if token.dep_ != 'ROOT':
                dependencies.append({
                    'child': token.text,
                    'child_pos': token.pos_,
                    'dep': token.dep_,
                    'parent': token.head.text,
                    'parent_pos': token.head.pos_
                })

        return dependencies
    except:
        return []

from tqdm.auto import tqdm

# Dictionnaire pour stocker les statistiques par langue
lang_dep_stats = {}
all_dependencies = defaultdict(lambda: defaultdict(int))

print("Extraction des dépendances syntaxiques par langue...")
print()

for lang_code in nlp_models.keys():
    print(f"Traitement du {lang_code.upper()}...")
    nlp = nlp_models[lang_code]

    lang_reviews = df_sample[df_sample['lang_code'] == lang_code]
    all_deps_lang = []
    all_pos_tags = []
    all_dep_types = []

    progress = tqdm(
        lang_reviews.iterrows(),
        total=len(lang_reviews),
        desc=f"Annotation {lang_code.upper()}",
        leave=False
    )

    for idx, row in progress:
        if pd.isna(row['comments']):
            continue

        dependencies = extract_dependencies(str(row['comments']), nlp)
        all_deps_lang.extend(dependencies)

        for dep in dependencies:
            all_pos_tags.append(dep['child_pos'])
            all_dep_types.append(dep['dep'])

    lang_dep_stats[lang_code] = {
        'total_tokens': len(all_pos_tags),
        'pos_distribution': Counter(all_pos_tags),
        'dep_distribution': Counter(all_dep_types),
        'all_dependencies': all_deps_lang
    }

    print(f"  ✓ {len(all_deps_lang)} relations de dépendance extraites")
    print(f"  ✓ {len(all_pos_tags)} tokens analysés")
    print()

print("Extraction terminée!")

Extraction des dépendances syntaxiques par langue...

Traitement du FR...


Annotation FR:   0%|          | 0/1673 [00:00<?, ?it/s]

  ✓ 63053 relations de dépendance extraites
  ✓ 63053 tokens analysés

Traitement du EN...


Annotation EN:   0%|          | 0/6680 [00:00<?, ?it/s]

  ✓ 391772 relations de dépendance extraites
  ✓ 391772 tokens analysés

Traitement du ES...


...

  ✓ 25812 relations de dépendance extraites
  ✓ 25812 tokens analysés

Traitement du PT...


Annotation PT:   0%|          | 0/270 [00:00<?, ?it/s]

  ✓ 14435 relations de dépendance extraites
  ✓ 14435 tokens analysés

Traitement du IT...


Annotation IT:   0%|          | 0/275 [00:00<?, ?it/s]

  ✓ 15337 relations de dépendance extraites
  ✓ 15337 tokens analysés

Extraction terminée!


## 8 Export des annotations en CSV (avec colonne id)

Cette étape exporte les annotations token par token vers un CSV dans `data/`.
Le format inclut l’identifiant de review (`id`), les informations de phrase et les champs syntaxiques (lemma, upos, head, deprel, etc.).

In [8]:
from pathlib import Path

out_csv = Path('data/results_reviews_select_sample_annotations.csv')
out_csv.parent.mkdir(parents=True, exist_ok=True)

rows = []
sent_count = 0
token_count = 0

for review_idx, row in df_sample.iterrows():
    lang_code = row['lang_code']
    if lang_code not in nlp_models:
        continue

    text = '' if pd.isna(row['comments']) else str(row['comments']).strip()
    if not text:
        continue

    if 'id' in df_sample.columns and pd.notna(row['id']):
        try:
            review_id = str(int(row['id']))
        except Exception:
            review_id = str(row['id'])
    else:
        review_id = str(review_idx)

    doc = nlp_models[lang_code](text[:2000])

    for sent in doc.sents:
        sent_text = sent.text.strip()
        if not sent_text:
            continue

        sent_count += 1

        for token in sent:
            token_id = token.i - sent.start + 1
            head = 0 if token.dep_ == 'ROOT' else (token.head.i - sent.start + 1)

            rows.append({
                'id': review_id,
                'review_index': review_idx,
                'lang': lang_code,
                'sent_id': sent_count,
                'sent_text': sent_text,
                'token_id': token_id,
                'form': token.text,
                'lemma': token.lemma_ if token.lemma_ else '_',
                'upos': token.pos_ if token.pos_ else '_',
                'xpos': token.tag_ if token.tag_ else '_',
                'feats': '_',
                'head': head,
                'deprel': token.dep_ if token.dep_ else '_',
                'deps': '_',
                'misc': '_'
            })
            token_count += 1

ann_df = pd.DataFrame(rows)
ann_df.to_csv(out_csv, index=False, encoding='utf-8')

print(f"Fichier CSV exporté: {out_csv}")
print(f"Lignes d'annotation: {len(ann_df)}")
print(f"Phrases exportées: {sent_count}")
print(f"Tokens exportés: {token_count}")

ann_df.head()

Fichier CSV exporté: data\results_reviews_select_sample_annotations.csv
Lignes d'annotation: 579440
Phrases exportées: 42641
Tokens exportés: 579440


,id,review_index,lang,sent_id,sent_text,token_id,form,lemma,upos,xpos,feats,head,deprel,deps,misc
0,916489788146905600,0,it,1,"Alloggio confortevole e pratico, dotato di tut...",1,Alloggio,Alloggio,VERB,V,_,0,ROOT,_,_
1,916489788146905600,0,it,1,"Alloggio confortevole e pratico, dotato di tut...",2,confortevole,confortevole,ADJ,A,_,1,amod,_,_
2,916489788146905600,0,it,1,"Alloggio confortevole e pratico, dotato di tut...",3,e,e,CCONJ,CC,_,4,cc,_,_
3,916489788146905600,0,it,1,"Alloggio confortevole e pratico, dotato di tut...",4,pratico,pratico,ADJ,A,_,2,conj,_,_
4,916489788146905600,0,it,1,"Alloggio confortevole e pratico, dotato di tut...",5,",",",",PUNCT,FF,_,1,punct,_,_


In [9]:
# Résumé textuel des dépendances (sans graphique)
print("Top dépendances par langue:")
for lang_code, stats in sorted(lang_dep_stats.items()):
    top10 = stats['dep_distribution'].most_common(10)
    print(f"\n{lang_code.upper()}:")
    if not top10:
        print("  Aucune dépendance extraite")
        continue
    for dep_name, dep_count in top10:
        print(f"  - {dep_name}: {dep_count}")

Top dépendances par langue:

DE:
  - nk: 6472
  - mo: 4690
  - punct: 3653
  - sb: 2483
  - cj: 1737
  - cd: 1171
  - pd: 1120
  - oc: 1051
  - oa: 996
  - mnr: 720

EN:
  - punct: 48382
  - nsubj: 34606
  - det: 34099
  - prep: 33922
  - pobj: 32127
  - advmod: 29282
  - amod: 26735
  - conj: 22935
  - cc: 20484
  - dobj: 14005

ES:
  - punct: 3412
  - det: 3297
  - case: 2767
  - advmod: 2187
  - amod: 1664
  - conj: 1479
  - nsubj: 1337
  - cc: 1322
  - nmod: 1298
  - obj: 1297

FR:
  - punct: 8313
  - case: 7320
  - det: 6495
  - advmod: 5751
  - nsubj: 4171
  - nmod: 3849
  - amod: 3727
  - conj: 3603
  - cc: 2958
  - obj: 1805

IT:
  - punct: 1943
  - case: 1913
  - det: 1414
  - conj: 1160
  - advmod: 1060
  - amod: 1018
  - obl: 957
  - nmod: 877
  - cc: 871
  - nsubj: 707

PT:
  - punct: 2179
  - case: 1615
  - det: 1371
  - advmod: 1101
  - conj: 990
  - nsubj: 856
  - amod: 794
  - cc: 766
  - obj: 754
  - nmod: 743


## 9. Synthèse textuelle des POS tags par langue

La section suivante affiche une synthèse textuelle des POS tags les plus fréquents par langue.
Elle offre une lecture rapide des catégories morpho-syntaxiques dominantes.

In [10]:
# Résumé textuel des POS tags (sans graphique)
print("Top POS tags par langue:")
for lang_code, stats in sorted(lang_dep_stats.items()):
    top10 = stats['pos_distribution'].most_common(10)
    print(f"\n{lang_code.upper()}:")
    if not top10:
        print("  Aucun POS tag extrait")
        continue
    for pos_name, pos_count in top10:
        print(f"  - {pos_name}: {pos_count}")

Top POS tags par langue:

DE:
  - ADV: 4429
  - NOUN: 4270
  - PUNCT: 3628
  - DET: 2721
  - ADP: 2272
  - PRON: 2115
  - VERB: 1493
  - ADJ: 1341
  - CCONJ: 1193
  - PROPN: 843

EN:
  - NOUN: 67403
  - PUNCT: 47655
  - ADJ: 45093
  - ADP: 36202
  - DET: 35008
  - PRON: 33118
  - ADV: 28294
  - VERB: 24991
  - CCONJ: 20416
  - PROPN: 16801

ES:
  - NOUN: 4529
  - ADP: 3406
  - PUNCT: 3395
  - DET: 3097
  - ADJ: 2489
  - ADV: 2092
  - VERB: 1572
  - PRON: 1385
  - CCONJ: 1313
  - AUX: 1266

FR:
  - NOUN: 10716
  - ADP: 8462
  - PUNCT: 8170
  - DET: 6491
  - ADV: 5973
  - ADJ: 5209
  - PRON: 3976
  - VERB: 3831
  - AUX: 3500
  - CCONJ: 2945

IT:
  - NOUN: 2826
  - ADP: 2096
  - PUNCT: 1920
  - DET: 1577
  - ADJ: 1461
  - ADV: 1136
  - AUX: 987
  - VERB: 981
  - CCONJ: 861
  - PRON: 610

PT:
  - NOUN: 2719
  - PUNCT: 2172
  - ADP: 1648
  - DET: 1415
  - ADV: 1195
  - ADJ: 1154
  - VERB: 1061
  - CCONJ: 747
  - AUX: 687
  - PRON: 578


## 10. Tableau récapitulatif syntaxique par langue

Ici, un tableau de synthèse rassemble les indicateurs clés par langue (tokens, relations, types uniques, dominante POS/DEP).
Ce tableau sert de vue consolidée des résultats intermédiaires.

In [11]:
# Créer un tableau récapitulatif
summary_data = []

for lang_code, stats in lang_dep_stats.items():
    top_dep = stats['dep_distribution'].most_common(1)[0][0] if stats['dep_distribution'] else 'N/A'
    top_pos = stats['pos_distribution'].most_common(1)[0][0] if stats['pos_distribution'] else 'N/A'
    
    summary_data.append({
        'Langue': lang_code.upper(),
        'Tokens analysés': stats['total_tokens'],
        'Relations de dépendance': len(stats['all_dependencies']),
        'Types de dépendances uniques': len(stats['dep_distribution']),
        'POS tags uniques': len(stats['pos_distribution']),
        'Dépendance dominante': top_dep,
        'POS dominant': top_pos
    })

summary_df = pd.DataFrame(summary_data)
print("\nRésumé des statistiques syntaxiques par langue:")
print(summary_df.to_string(index=False))


Résumé des statistiques syntaxiques par langue:
Langue  Tokens analysés  Relations de dépendance  Types de dépendances uniques  POS tags uniques Dépendance dominante POS dominant
    FR            63053                    63053                            35                17                punct         NOUN
    EN           391772                   391772                            44                18                punct         NOUN
    ES            26406                    26406                            30                17                punct         NOUN
    DE            25812                    25812                            37                16                   nk          ADV
    PT            14435                    14435                            35                15                punct         NOUN
    IT            15337                    15337                            39                17                punct         NOUN


## 12. Analyse des ROOT par langue (texte)

Cette partie isole les tokens `ROOT` pour observer les verbes principaux les plus fréquents selon la langue.
L’objectif est de caractériser la structure prédicative dominante des commentaires.

In [12]:
# Analyser les verbes principaux (ROOT) et structures
print("\n" + "="*60)
print("ANALYSE DES VERBES PRINCIPAUX (ROOT) PAR LANGUE")
print("="*60)

for lang_code in sorted(nlp_models.keys()):
    nlp = nlp_models[lang_code]
    lang_reviews = df_sample[df_sample['lang_code'] == lang_code]

    root_verbs = []

    for idx, row in lang_reviews.iterrows():
        if pd.isna(row['comments']):
            continue

        try:
            doc = nlp(str(row['comments'])[:1000])
            for token in doc:
                if token.dep_ == 'ROOT':
                    root_verbs.append(token.text)
        except:
            pass

    print(f"\n{lang_code.upper()} - Verbes principaux les plus fréquents:")
    verb_freq = Counter(root_verbs)
    for verb, count in verb_freq.most_common(10):
        print(f"  '{verb}' : {count} fois")


ANALYSE DES VERBES PRINCIPAUX (ROOT) PAR LANGUE

DE - Verbes principaux les plus fréquents:
  'ist' : 461 fois
  'war' : 280 fois
  'hat' : 126 fois
  'haben' : 115 fois
  'waren' : 82 fois
  'sind' : 69 fois
  'hatten' : 54 fois
  'kommen' : 40 fois
  'liegt' : 40 fois
  'gibt' : 39 fois

EN - Verbes principaux les plus fréquents:
  'was' : 5734 fois
  'is' : 4572 fois
  'had' : 1068 fois
  'recommend' : 1019 fois
  'were' : 879 fois
  'stay' : 723 fois
  'location' : 679 fois
  'are' : 637 fois
  'loved' : 498 fois
  'place' : 491 fois

ES - Verbes principaux les plus fréquents:
  'amable' : 49 fois
  'ubicación' : 49 fois
  'lugar' : 43 fois
  'ubicado' : 36 fois
  'excelente' : 25 fois
  'departamento' : 24 fois
  'recomendable' : 23 fois
  'tiene' : 21 fois
  'atenta' : 20 fois
  'lindo' : 20 fois

FR - Verbes principaux les plus fréquents:
  'recommande' : 259 fois
  'Merci' : 227 fois
  'passé' : 202 fois
  'appartement' : 193 fois
  'Appartement' : 154 fois
  'Logement' : 105 

## 15. Conclusions et résultats clés

La conclusion produit un résumé global: couverture linguistique, tableaux POS/DEP dominants et bilan temps/énergie/CO2.
Cette section fournit la sortie finale interprétable du notebook.

In [15]:
print("\n" + "="*70)
print("RÉSUMÉ FINAL DE L'ANALYSE DES DÉPENDANCES SYNTAXIQUES")
print("="*70)

print(f"\n1. COUVERTURE LINGUISTIQUE")
print(f"   - Total de reviews analysées: {len(df)}")
print(f"   - Reviews retenues pour annotation: {len(df_sample)}")
langs_upper = [lang.upper() for lang in sorted(nlp_models.keys())]
print(f"   - Langues analysées: {', '.join(langs_upper)}")

# Tableaux de fréquences relatives (base: lignes = langues)
pos_abs_df = pd.DataFrame(
    {lang.upper(): stats['pos_distribution'] for lang, stats in sorted(lang_dep_stats.items())}
).T.fillna(0)
if not pos_abs_df.empty:
    pos_abs_df = pos_abs_df.loc[:, pos_abs_df.sum(axis=0) > 0]
pos_rel_df = pos_abs_df.div(pos_abs_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)


dep_abs_df = pd.DataFrame(
    {lang.upper(): stats['dep_distribution'] for lang, stats in sorted(lang_dep_stats.items())}
).T.fillna(0)
if not dep_abs_df.empty:
    dep_abs_df = dep_abs_df.loc[:, dep_abs_df.sum(axis=0) > 0]
dep_rel_df = dep_abs_df.div(dep_abs_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

# Transposition demandée: lignes = POS/deprel, colonnes = langues
pos_rel_t = pos_rel_df.T
pos_rel_t.columns.name = "langue"

dep_rel_t = dep_rel_df.T
dep_rel_t.columns.name = "langue"

print("\n2. RÉPARTITION RELATIVE DES TOKENS PAR POS (LIGNES=POS, COLONNES=LANGUES)")
print(pos_rel_t.round(4).to_string())

print("\n3. RÉPARTITION RELATIVE DES DÉPENDANCES (LIGNES=DEPREL, COLONNES=LANGUES)")
print(dep_rel_t.round(4).to_string())

# Bilan global temps/carbone (à la toute fin)
elapsed_sec_total = time.perf_counter() - SCRIPT_START
energy_kwh_total = (POWER_WATTS / 1000) * (elapsed_sec_total / 3600)
energy_wh_total = energy_kwh_total * 1000
co2_kg_total = energy_kwh_total * FR_GRID_KGCO2_PER_KWH
co2_g_total = co2_kg_total * 1000

print("\n4. BILAN GLOBAL TEMPS ET CARBONE")
print(f"   - Temps total de traitement: {elapsed_sec_total:.2f} s")
print(f"   - Énergie estimée ({POWER_WATTS}W): {energy_wh_total:.2f} Wh")
print(f"   - Émissions CO2 estimées (mix FR): {co2_g_total:.2f} gCO2e")

print(f"\n{'='*70}")
print(f"TEMPS DE CALCUL FINAL: {elapsed_sec_total:.2f} s")
print(f"ENERGIE FINALE: {energy_wh_total:.2f} Wh")
print(f"CO2 FINAL: {co2_g_total:.2f} gCO2e")


RÉSUMÉ FINAL DE L'ANALYSE DES DÉPENDANCES SYNTAXIQUES

1. COUVERTURE LINGUISTIQUE
   - Total de reviews analysées: 530818
   - Reviews retenues pour annotation: 10000
   - Langues analysées: DE, EN, ES, FR, IT, PT

2. RÉPARTITION RELATIVE DES TOKENS PAR POS (LIGNES=POS, COLONNES=LANGUES)
langue      DE      EN      ES      FR      IT      PT
DET     0.1054  0.0894  0.1173  0.1029  0.1028  0.0980
CCONJ   0.0462  0.0521  0.0497  0.0467  0.0561  0.0517
PRON    0.0819  0.0845  0.0525  0.0631  0.0398  0.0400
VERB    0.0578  0.0638  0.0595  0.0608  0.0640  0.0735
ADJ     0.0520  0.1151  0.0943  0.0826  0.0953  0.0799
NOUN    0.1654  0.1720  0.1715  0.1700  0.1843  0.1884
ADP     0.0880  0.0924  0.1290  0.1342  0.1367  0.1142
PUNCT   0.1406  0.1216  0.1286  0.1296  0.1252  0.1505
ADV     0.1716  0.0722  0.0792  0.0947  0.0741  0.0828
SCONJ   0.0083  0.0125  0.0162  0.0062  0.0076  0.0205
PART    0.0102  0.0259  0.0000  0.0000  0.0000  0.0000
AUX     0.0263  0.0356  0.0479  0.0555  0.0644  0.